In [ ]:
# ============================================================================
# IV CRUSH SCREENER - PRECISION ENTRY (TUE-MON CYCLE)
# ============================================================================

!pip install yfinance pandas numpy tqdm -q

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, time as dtime
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import warnings
import time

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

class Config:
    # Run settings
    INCLUDE_TODAY_IF_BEFORE_OPEN = True
    MARKET_OPEN_TIME = 9
    
    # Filters
    MIN_PRICE = 5
    MAX_PRICE = 2000
    
    MIN_IV_PERCENTILE = 30
    APPLY_IV_FILTER = False
    
    TOP_N = 10
    MAX_SECTOR_CONCENTRATION = 2


# ============================================================================
# WATCHLIST
# ============================================================================

raw_tickers = """
A AA AAL AAP AAPL ABBV ABNB ABT ACN ADBE ADI ADM ADP ADSK AEP AES AFL AFRM AGNC AI AIG AKAM ALB ALK ALL AMAT AMD AMGN AMRN AMZN APA APH APTV AVGO AXP BA BABA BAC BAX BBY BEN BIDU BIIB BK BKR BMY BP BSX BX BYND C CAH CAT CB CCI CCJ CCL CF CFG CHWY CI CL CLF CLX CMCSA CME CNC CNP COF COIN COP COST CPB CPRI CRM CRON CRWD CSCO CSX CTVA CVNA CVS CVX CZR D DAL DD DE DELL DHI DHR DIS DKNG DLR DOCU DOW DRI DVN DXC EA EBAY ED EIX EL EMR ENPH EOG EQR EQT ETSY EVRG EW EXC EXPE F FANG FAST FCX FDX FE FITB FOXA FSLR FTI FTV GD GE GILD GLW GM GME GOOG GOOGL GPRO GPS GS HAL HBAN HCA HD HIG HLT HOG HOLX HON HPE HPQ IBM ICE INTC IP IRM IVZ JCI JD JNJ JPM K KHC KMI KO KR KSS LEN LI LKQ LLY LNC LOW LUMN LUV LVS LYB LYFT MA MAR MARA MAS MCD MCHP MDLZ MDT MET META MGM MMM MO MOS MPC MRK MRNA MRVL MS MSFT MSTR MTB MTCH MU NCLH NEE NEM NET NFLX NKE NOW NRG NTAP NTES NVAX NVDA NWL NWSA O OKE OMC ON ORCL OXY PBR PDD PENN PEP PFE PFG PG PGR PINS PLTR PNC PPL PRU PSX PTON PYPL QCOM RBLX RCL RF RIG RIOT RIVN ROKU ROST RTX RUN SBUX SCHW SEDG SHOP SIRI SLB SMCI SNAP SNOW SO SOFI SPG STX SWKS SYF SYY T TAP TCOM TDOC TFC TGT TJX TMO TMUS TPR TRIP TRV TSLA TSM TSN TT TTD TTWO TXN U UA UAA UAL UBER ULTA UNH UNP UPS UPST URBN USB V VALE VFC VTR VZ WAB WBD WDC WFC WM WMB WMT WU WY WYNN XEL YELP ZION ZM
"""

WATCHLIST = [ticker.strip() for ticker in raw_tickers.split() if ticker.strip()]

ETF_EXCLUSIONS = {
    'DIA', 'SPY', 'QQQ', 'IWM', 'EEM', 'EFA', 'GLD', 'SLV', 'GDX', 'HYG', 
    'IBB', 'IEF', 'IYR', 'KRE', 'LQD', 'OIH', 'SMH', 'TLT', 'UNG', 'USO', 
    'VGK', 'VXX', 'XBI', 'XHB', 'XLB', 'XLC', 'XLE', 'XLF', 'XLI', 'XLK', 
    'XLP', 'XLRE', 'XLU', 'XLV', 'XLY', 'XOP', 'XRT', 'SOXL', 'SOXS', 
    'SPXL', 'SPXS', 'SQQQ', 'TQQQ', 'UVXY', 'JETS', 'BITO', 'IBIT', 'KWEB', 
    'YINN', 'FXE', 'FXI', 'TBT', 'SCHD'
}

WATCHLIST = [ticker for ticker in WATCHLIST if ticker not in ETF_EXCLUSIONS]
print(f"📊 Loaded {len(WATCHLIST)} individual stocks\n")


# ============================================================================
# TRADING CALENDAR
# ============================================================================

class TradingCalendar:
    @staticmethod
    def is_trading_day(date):
        if isinstance(date, datetime):
            date = date.date()
        return date.weekday() < 5
    
    @staticmethod
    def get_next_trading_day(date):
        if isinstance(date, datetime):
            date = date.date()
        while not TradingCalendar.is_trading_day(date):
            date += timedelta(days=1)
        return date
    
    @staticmethod
    def get_previous_trading_day(date):
        if isinstance(date, datetime):
            date = date.date()
        date -= timedelta(days=1)
        while not TradingCalendar.is_trading_day(date):
            date -= timedelta(days=1)
        return date
    
    @staticmethod
    def add_trading_days(date, days):
        if isinstance(date, datetime):
            date = date.date()
        if days >= 0:
            while days > 0:
                date += timedelta(days=1)
                if TradingCalendar.is_trading_day(date):
                    days -= 1
        else:
            while days < 0:
                date -= timedelta(days=1)
                if TradingCalendar.is_trading_day(date):
                    days += 1
        return date
    
    @staticmethod
    def calculate_precise_entry_date(earnings_date, earnings_time):
        """
        Logic:
        - If AMC (After Market Close): Enter SAME DAY (30 mins before close)
        - If BMO (Before Market Open): Enter PREVIOUS TRADING DAY (30 mins before close)
        """
        if isinstance(earnings_date, datetime):
            earnings_date = earnings_date.date()
            
        if earnings_time == 'AMC':
            return earnings_date
        else: # BMO
            return TradingCalendar.get_previous_trading_day(earnings_date)


# ============================================================================
# ENTRY WINDOW (FIXED FOR 6 DAY CYCLE: TUE-MON)
# ============================================================================

class EntryWindow:
    def __init__(self, run_datetime=None):
        self.run_datetime = run_datetime or datetime.now()
        self.run_date = self.run_datetime.date()
        self.run_hour = self.run_datetime.hour
        self.calendar = TradingCalendar()
        
        self.is_before_open = self.run_hour < Config.MARKET_OPEN_TIME
        
        # 1. Determine START of Entry Window
        if self.run_date.weekday() == 1 and self.is_before_open and Config.INCLUDE_TODAY_IF_BEFORE_OPEN:
            self.first_entry_date = self.run_date
        else:
            self.first_entry_date = self.calendar.get_next_trading_day(self.run_date)
        
        # 2. Determine END of Entry Window (STRICT 6-DAY LIMIT)
        self.last_entry_date = self.first_entry_date + timedelta(days=6)
    
    def print_info(self):
        print(f"{'='*75}")
        print(f"  📅 SCAN CONFIGURATION (6-Day Cycle: Tue -> Mon)")
        print(f"{'='*75}")
        print(f"""
  Run Date/Time:        {self.run_datetime.strftime('%A, %B %d, %Y at %I:%M %p')}
  Before Market Open:   {'✅ Yes' if self.is_before_open else '❌ No'}
  
  Entry Window:         {self.first_entry_date.strftime('%a %m/%d')} - {self.last_entry_date.strftime('%a %m/%d')}
  Entry Logic:          30 mins before Close of the Trading Session before Earnings
        """)


# ============================================================================
# DATA FETCHERS
# ============================================================================

def get_earnings_date(ticker):
    """Get earnings date using multiple methods"""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        
        earnings_dt = None
        earnings_time = 'AMC' # Default assumption
        
        if info.get('earningsTimestamp'):
            earnings_dt = datetime.fromtimestamp(info['earningsTimestamp'])
            earnings_time = 'BMO' if earnings_dt.hour < 12 else 'AMC'
        
        if earnings_dt is None:
            try:
                calendar = stock.calendar
                if calendar and 'Earnings Date' in calendar:
                    earnings_dates = calendar['Earnings Date']
                    if isinstance(earnings_dates, list) and len(earnings_dates) > 0:
                        ed = earnings_dates[0]
                        if hasattr(ed, 'year'):
                            earnings_dt = datetime.combine(ed, datetime.min.time()) if not isinstance(ed, datetime) else ed
            except:
                pass
        
        if earnings_dt is None:
            try:
                df = stock.earnings_dates
                if df is not None and len(df) > 0:
                    dt = df.index[0]
                    if hasattr(dt, 'tz_localize'): dt = dt.tz_localize(None)
                    elif hasattr(dt, 'tzinfo') and dt.tzinfo is not None: dt = dt.replace(tzinfo=None)
                    if hasattr(dt, 'to_pydatetime'): dt = dt.to_pydatetime()
                    
                    if isinstance(dt, datetime) and dt.date() >= datetime.now().date():
                        earnings_dt = dt
            except:
                pass
        
        if earnings_dt:
            return {
                'ticker': ticker,
                'earnings_datetime': earnings_dt,
                'earnings_date': earnings_dt.date() if isinstance(earnings_dt, datetime) else earnings_dt,
                'earnings_time': earnings_time
            }
        
        return None
        
    except:
        return None


def check_entry_window(earnings_info, window):
    """Check if PRECISE entry date falls within actionable entry window"""
    earnings_date = earnings_info['earnings_date']
    earnings_time = earnings_info['earnings_time']
    
    # NEW LOGIC: Calculate precise entry date based on AMC/BMO
    entry_date = TradingCalendar.calculate_precise_entry_date(earnings_date, earnings_time)
    
    if earnings_time == 'BMO':
        exit_date = earnings_date # Close same day morning
    else:
        exit_date = TradingCalendar.add_trading_days(earnings_date, 1) # Close next day morning
    
    in_window = window.first_entry_date <= entry_date <= window.last_entry_date
    
    return {
        **earnings_info,
        'entry_date': entry_date,
        'exit_date': exit_date,
        'in_window': in_window,
        'days_until_entry': (entry_date - datetime.now().date()).days
    }


def get_expected_move(ticker, earnings_date):
    """Get expected move from options straddle and SECTOR data"""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        sector = info.get('sector', 'Unknown')
        
        price = info.get('regularMarketPrice') or info.get('currentPrice')
        if not price:
            hist = stock.history(period='5d')
            price = hist['Close'].iloc[-1] if len(hist) > 0 else None
        if not price: return None
        
        expirations = stock.options
        if not expirations: return None
        
        target_exp = None
        for exp in expirations:
            exp_dt = datetime.strptime(exp, '%Y-%m-%d')
            if exp_dt.date() >= earnings_date:
                target_exp = exp
                break
        
        if not target_exp: target_exp = expirations[0]
        
        chain = stock.option_chain(target_exp)
        calls, puts = chain.calls, chain.puts
        
        if len(calls) == 0 or len(puts) == 0: return None
        
        def get_mid(row):
            if row['bid'] > 0 and row['ask'] > 0: return (row['bid'] + row['ask']) / 2
            return row['lastPrice']
        
        calls['dist'] = abs(calls['strike'] - price)
        atm_strike = calls.loc[calls['dist'].idxmin(), 'strike']
        
        atm_call = calls[calls['strike'] == atm_strike].iloc[0]
        atm_put = puts[puts['strike'] == atm_strike].iloc[0]
        
        call_mid = get_mid(atm_call)
        put_mid = get_mid(atm_put)
        straddle = call_mid + put_mid
        expected_move_pct = (straddle / price) * 100
        
        call_iv = (atm_call.get('impliedVolatility') or 0) * 100
        put_iv = (atm_put.get('impliedVolatility') or 0) * 100
        avg_iv = (call_iv + put_iv) / 2 if (call_iv > 0 and put_iv > 0) else None
        
        wing_distance = straddle
        long_call_candidates = calls[calls['strike'] >= atm_strike + wing_distance * 0.8]
        long_put_candidates = puts[puts['strike'] <= atm_strike - wing_distance * 0.8]
        
        if len(long_call_candidates) == 0: long_call_candidates = calls[calls['strike'] > atm_strike].head(3)
        if len(long_put_candidates) == 0: long_put_candidates = puts[puts['strike'] < atm_strike].tail(3)
        
        if len(long_call_candidates) > 0 and len(long_put_candidates) > 0:
            long_call = long_call_candidates.iloc[0]
            long_put = long_put_candidates.iloc[-1]
            long_call_price = get_mid(long_call)
            long_put_price = get_mid(long_put)
            credit = (call_mid + put_mid) - (long_call_price + long_put_price)
            max_loss = max(long_call['strike'] - atm_strike, atm_strike - long_put['strike']) - credit
        else:
            credit = None
            max_loss = None
        
        return {
            'sector': sector,
            'price': price,
            'atm_strike': atm_strike,
            'straddle_price': straddle,
            'expected_move_pct': expected_move_pct,
            'avg_iv': avg_iv,
            'expiry': target_exp,
            'credit': credit,
            'max_loss': max_loss
        }
    except:
        return None


def get_historical_moves(ticker):
    """Get historical earnings moves"""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period='2y')
        
        if len(hist) < 100: return None
        
        hist['return'] = hist['Close'].pct_change().abs() * 100
        hist['quarter'] = hist.index.to_period('Q')
        quarterly_max = hist.groupby('quarter')['return'].max()
        moves = quarterly_max.dropna().tail(8).tolist()
        
        if len(moves) >= 4:
            return {
                'avg': np.mean(moves),
                'median': np.median(moves),
                'max': np.max(moves),
                'moves': moves
            }
        return None
    except:
        return None


def get_iv_percentile(ticker):
    """Calculate IV percentile"""
    try:
        stock = yf.Ticker(ticker)
        hist = stock.history(period='1y')
        if len(hist) < 60: return None
        
        hist['ret'] = np.log(hist['Close'] / hist['Close'].shift(1))
        hist['hv20'] = hist['ret'].rolling(20).std() * np.sqrt(252) * 100
        current_hv = hist['hv20'].iloc[-1]
        hv_series = hist['hv20'].dropna()
        if len(hv_series) < 20: return None
        return (hv_series < current_hv).sum() / len(hv_series) * 100
    except:
        return None


# ============================================================================
# MAIN SCREENER
# ============================================================================

def run_screener():
    print(f"""
    ╔═══════════════════════════════════════════════════════════════╗
    ║   IV CRUSH SCREENER - PRECISION ENTRY (TUE-MON)              ║
    ╚═══════════════════════════════════════════════════════════════╝
    """)
    
    window = EntryWindow()
    window.print_info()
    
    # STEP 1 & 2: EARNINGS & WINDOW
    print(f"\nScanning {len(WATCHLIST)} stocks for earnings...")
    in_window = []
    
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(get_earnings_date, t): t for t in WATCHLIST}
        for future in tqdm(as_completed(futures), total=len(WATCHLIST), desc="Earnings Dates"):
            res = future.result()
            if res:
                check = check_entry_window(res, window)
                if check['in_window']:
                    in_window.append(check)
    
    if not in_window:
        print("\n❌ No stocks in entry window!")
        return None
    
    print(f"\n✅ Found {len(in_window)} stocks matching entry logic.")

    # STEP 3: OPTIONS & SECTOR
    with_options = []
    for s in tqdm(in_window, desc="Options & Sector Data"):
        opt = get_expected_move(s['ticker'], s['earnings_date'])
        if opt: with_options.append({**s, **opt})
        time.sleep(0.05)
        
    if not with_options:
        print("\n❌ No stocks with options data!")
        return None

    # STEP 4: HISTORICAL
    with_history = []
    for s in tqdm(with_options, desc="Historical Moves"):
        hist = get_historical_moves(s['ticker'])
        if hist:
            move_ratio = hist['avg'] / s['expected_move_pct']
            beat_rate = sum(1 for m in hist['moves'] if m < s['expected_move_pct']) / len(hist['moves']) * 100
            with_history.append({
                **s, 'hist_avg_move': hist['avg'], 'move_ratio': move_ratio, 'beat_rate': beat_rate
            })
        else:
            with_history.append({
                **s, 'hist_avg_move': None, 'move_ratio': None, 'beat_rate': None
            })
        time.sleep(0.02)

    # STEP 5: IV PERCENTILE
    final_data = []
    for s in tqdm(with_history, desc="IV Percentile"):
        iv_pct = get_iv_percentile(s['ticker'])
        final_data.append({**s, 'iv_percentile': iv_pct})
        time.sleep(0.02)
        
    # STEP 6: SCORE & DIVERSIFY
    print(f"\n🎯 SCORING & APPLYING SECTOR DIVERSITY (Max {Config.MAX_SECTOR_CONCENTRATION} per sector)")
    
    for s in final_data:
        score = 15 # Base
        if s['iv_percentile']: score += (s['iv_percentile'] / 100) * 25
        else: score += 10
        if s['move_ratio']: score += max(0, min(1, 1.4 - s['move_ratio'])) * 35
        else: score += 15
        if s['beat_rate']: score += (s['beat_rate'] / 100) * 25
        else: score += 12
        s['score'] = round(score, 1)
        
    final_data.sort(key=lambda x: x['score'], reverse=True)
    
    selected_stocks = []
    sector_counts = {}
    deferred_stocks = []
    
    for s in final_data:
        sector = s['sector']
        current_count = sector_counts.get(sector, 0)
        
        if current_count < Config.MAX_SECTOR_CONCENTRATION:
            selected_stocks.append(s)
            sector_counts[sector] = current_count + 1
        else:
            deferred_stocks.append(s)
            
        if len(selected_stocks) >= Config.TOP_N:
            break
    
    if len(selected_stocks) < Config.TOP_N:
        needed = Config.TOP_N - len(selected_stocks)
        selected_stocks.extend(deferred_stocks[:needed])
        
    return selected_stocks


# ============================================================================
# OUTPUT
# ============================================================================

def print_results_with_rules(top_stocks):
    if not top_stocks: return
    
    print(f"\n{'='*75}")
    print(f"  🏆 TOP {len(top_stocks)} DIVERSIFIED STOCKS")
    print(f"{'='*75}\n")
    
    tickers = [s['ticker'] for s in top_stocks]
    print(f"  Bot Input List: {', '.join(tickers)}\n")
    
    by_entry = {}
    for s in top_stocks:
        entry_str = s['entry_date'].strftime('%Y-%m-%d')
        if entry_str not in by_entry: by_entry[entry_str] = []
        by_entry[entry_str].append(s)
    
    for entry_str in sorted(by_entry.keys()):
        stocks = by_entry[entry_str]
        entry_dt = datetime.strptime(entry_str, '%Y-%m-%d').date()
        print(f"  📅 ENTRY: {entry_dt.strftime('%A %b %d')} (Last 30 mins of session)")
        for s in stocks:
            print(f"    → {s['ticker']:<6} [{s['sector']}]")
            print(f"      Report: {s['earnings_date'].strftime('%m/%d')} {s['earnings_time']}")
            print(f"      Ratio: {s['move_ratio']:.2f}x (Exp: ±{s['expected_move_pct']:.1f}% vs Hist: ±{s['hist_avg_move']:.1f}%)")

    df = pd.DataFrame(top_stocks)
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    csv_file = f'iv_crush_precision_{timestamp}.csv'
    df.to_csv(csv_file, index=False)
    print(f"\n  ✅ Saved to: {csv_file}")
    
    print(f"\n{'='*75}")
    print(f"  📋 PRECISION BOT RULES")
    print(f"{'='*75}\n")
    print("""
  1. ENTRY TIMING (CRITICAL):
     • If Earnings is AMC (After Market): Enter TODAY (30 mins before close).
     • If Earnings is BMO (Before Market): Enter YESTERDAY (30 mins before close).
  
  2. STOP LOSS / RISK:
     • NO Stop Loss order (Gaps kill stops).
     • Risk limited by Position Size (Max 2.5% acct).
     • Hard Exit: Market Open + 10 mins (Win or Loss).
  
  3. DIVERSITY:
     • Max 2 stocks per sector enforced.
    """)

if __name__ == "__main__":
    results = run_screener()
    if results: print_results_with_rules(results)